# Lost in the Museum — CLIP embedding baseline

Cross-domain image retrieval baseline for the *Lost in the Museum* competition.

**Approach**

- Every one of the 20,000 images (HQ gallery + real-world queries + distractor "dummy" images) gets
  embedded with the **same** pretrained CLIP image encoder (`openai/clip-vit-base-patch32`, 512-d output).
  CLIP was contrastively trained on hundreds of millions of noisy web image–text pairs (including plenty
  of photographs of artwork taken in the wild), so its embedding space already generalizes reasonably
  well across the "pristine studio photo" vs. "blurry visitor snapshot" domain gap — much better than a
  plain ImageNet-classification backbone would.
- Embeddings are L2-normalized, as required for cosine-similarity scoring.
- Simple horizontal-flip test-time augmentation is applied to every image (not just queries, since we are
  never told which images are which) and the two views are averaged then re-normalized — this smooths out
  a bit of the noise from crops/glare/motion blur without needing to know which images are queries.
- The notebook **does not hardcode any dataset path**. It walks `/kaggle/input` at runtime, finds every
  image file and the provided `submission.csv` template, and builds the image list from the template
  (the actual source of truth for which 20,000 `image_name`s are expected and in what order). If anything
  doesn't line up (missing files, duplicate filenames in different folders, etc.) the notebook fails loudly
  with a diagnostic instead of silently producing a wrong submission.

**Before running**: in the Kaggle notebook editor, attach the competition dataset, turn on an accelerator
(GPU T4 x2 or P100) under Settings, and make sure Internet is enabled (needed the first time to download
the pretrained CLIP weights — after that they're cached).


## 1. Setup

In [ ]:
import os, sys, subprocess

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    import transformers  # noqa: F401
except ImportError:
    pip_install("transformers")

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 2. Config

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"  # 512-d image embeddings, matches the recommended D
BATCH_SIZE = 64
USE_FLIP_TTA = True          # average embedding of image + horizontal flip, then re-normalize
NUM_WORKERS = 2
OUTPUT_CSV = "submission.csv"
SEED = 42

import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 3. Auto-discover the dataset

We don't assume any fixed folder layout. We:

1. Recursively scan `/kaggle/input` for every image file (`.png`, `.jpg`, `.jpeg`, `.bmp`, `.webp`).
2. Recursively look for a `submission.csv` (the provided template) to get the authoritative list of
   `ID` / `image_name` values and column schema.
3. Build a `basename -> full path` index from step 1 and join it against the template from step 2.

If the template can't be found, we fall back to using every discovered image file directly (less safe,
but keeps the notebook runnable for exploration).


In [ ]:
INPUT_ROOT = "/kaggle/input"
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

def find_image_files(root):
    found = []
    for dirpath, _dirnames, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(IMAGE_EXTS):
                found.append(os.path.join(dirpath, fname))
    return found

def find_submission_templates(root):
    found = []
    for dirpath, _dirnames, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() == "submission.csv":
                found.append(os.path.join(dirpath, fname))
    return found

all_images = find_image_files(INPUT_ROOT)
print(f"Found {len(all_images)} image files under {INPUT_ROOT}")

from collections import Counter
dir_counts = Counter(os.path.dirname(p) for p in all_images)
print("\nImage files per directory (top 10):")
for d, c in sorted(dir_counts.items(), key=lambda kv: -kv[1])[:10]:
    print(f"  {c:6d}  {d}")

templates = find_submission_templates(INPUT_ROOT)
print(f"\nFound {len(templates)} submission.csv template(s):")
for t in templates:
    print(" ", t)


In [ ]:
# basename -> list of full paths (there should be exactly one match per basename;
# more than one means duplicate filenames living in different folders, which we
# need to know about rather than silently pick the wrong one)
from collections import defaultdict

basename_to_paths = defaultdict(list)
for p in all_images:
    basename_to_paths[os.path.basename(p)].append(p)

dupes = {b: ps for b, ps in basename_to_paths.items() if len(ps) > 1}
if dupes:
    print(f"WARNING: {len(dupes)} filenames appear in more than one folder, e.g.:")
    for b, ps in list(dupes.items())[:5]:
        print(f"  {b}: {ps}")
    print("Using the first match for each; re-check this list before trusting the final submission.")

basename_to_path = {b: ps[0] for b, ps in basename_to_paths.items()}
print(f"\nIndexed {len(basename_to_path)} unique image basenames.")


In [ ]:
import pandas as pd

if templates:
    # Prefer a template that is NOT itself inside a previous /kaggle/working output
    template_path = sorted(templates, key=len)[0]
    print("Using submission template:", template_path)
    template_df = pd.read_csv(template_path)
    print("Template columns:", list(template_df.columns)[:6], "..." if len(template_df.columns) > 6 else "")
    print("Template shape:", template_df.shape)

    assert "image_name" in template_df.columns, "Expected an 'image_name' column in the template."
    id_col = "ID" if "ID" in template_df.columns else "image_name"

    feature_cols = [c for c in template_df.columns if c.startswith("feature_")]
    EMBED_DIM = len(feature_cols) if feature_cols else 512
    print(f"\nTarget embedding dimension D = {EMBED_DIM} "
          f"({'from template header' if feature_cols else 'default, no feature_* columns in template'})")

    order_df = template_df[[id_col, "image_name"]].copy()
    order_df.columns = ["ID", "image_name"]
else:
    print("No submission.csv template found — falling back to all discovered images.")
    EMBED_DIM = 512
    order_df = pd.DataFrame({
        "ID": [os.path.basename(p) for p in all_images],
        "image_name": [os.path.basename(p) for p in all_images],
    })

print(f"\nExpecting {len(order_df)} rows in the final submission.")


In [ ]:
# Resolve every expected image_name to an actual file path, and fail loudly on anything missing.
order_df["path"] = order_df["image_name"].map(basename_to_path)

missing = order_df[order_df["path"].isna()]
if len(missing) > 0:
    print(f"ERROR: {len(missing)} expected image(s) were not found on disk, e.g.:")
    print(missing.head(10))
    raise FileNotFoundError(
        f"{len(missing)} images from the submission template could not be located under {INPUT_ROOT}. "
        "Check the directory listing printed above and adjust IMAGE_EXTS / INPUT_ROOT if needed."
    )

n_dupe_names = order_df["image_name"].duplicated().sum()
assert n_dupe_names == 0, f"{n_dupe_names} duplicate image_name values in the template — unexpected."

print("All expected images resolved successfully.")
order_df.head()


## 4. Dataset / DataLoader

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class ImageListDataset(Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"WARNING: failed to read {path} ({e}); using a blank gray fallback image.")
            img = Image.new("RGB", (224, 224), color=(127, 127, 127))
        pixel_values = self.preprocess(images=img, return_tensors="pt")["pixel_values"][0]
        return pixel_values, idx


## 5. Load CLIP

In [ ]:
from transformers import CLIPModel, CLIPImageProcessor

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_NAME)
model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()

# sanity check on the actual output dimension of this checkpoint
with torch.no_grad():
    dummy = torch.zeros(1, 3, processor.crop_size["height"], processor.crop_size["width"], device=DEVICE)
    dummy_out = model.get_image_features(pixel_values=dummy)
MODEL_DIM = dummy_out.shape[-1]
print(f"CLIP checkpoint image-embedding dim: {MODEL_DIM}")

if MODEL_DIM != EMBED_DIM:
    print(f"NOTE: template expected D={EMBED_DIM} but the model produces {MODEL_DIM}-d embeddings. "
          f"Using the model's native dimension ({MODEL_DIM}) for the actual submission.")
    EMBED_DIM = MODEL_DIM


## 6. Extract embeddings for all 20,000 images

In [ ]:
import torch.nn.functional as F

paths_in_order = order_df["path"].tolist()
dataset = ImageListDataset(paths_in_order, processor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

all_embeddings = torch.zeros(len(paths_in_order), EMBED_DIM, dtype=torch.float32)

with torch.no_grad():
    for pixel_values, idxs in loader:
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)

        feats = model.get_image_features(pixel_values=pixel_values)

        if USE_FLIP_TTA:
            flipped = torch.flip(pixel_values, dims=[-1])  # horizontal flip
            feats_flip = model.get_image_features(pixel_values=flipped)
            feats = (feats + feats_flip) / 2.0

        feats = F.normalize(feats, p=2, dim=-1)
        all_embeddings[idxs] = feats.float().cpu()

print("Embeddings tensor shape:", all_embeddings.shape)


## 7. Sanity checks

In [ ]:
assert all_embeddings.shape[0] == len(order_df), "Row count mismatch."
assert not torch.isnan(all_embeddings).any(), "NaNs in embeddings!"

norms = all_embeddings.norm(dim=1)
print("Embedding L2 norm stats: min=%.4f max=%.4f mean=%.4f" % (norms.min(), norms.max(), norms.mean()))
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-3), "Embeddings are not unit-normalized."

print("All sanity checks passed.")


## 8. Build and save submission.csv

In [ ]:
feature_cols = [f"feature_{i}" for i in range(EMBED_DIM)]
emb_df = pd.DataFrame(all_embeddings.numpy(), columns=feature_cols)

submission = pd.concat(
    [order_df[["ID", "image_name"]].reset_index(drop=True), emb_df],
    axis=1,
)

assert len(submission) == 20000, f"Expected 20000 rows, got {len(submission)}"
assert submission["image_name"].duplicated().sum() == 0, "Duplicate image_name in final submission."
assert submission["image_name"].isna().sum() == 0, "Missing image_name values."

submission.to_csv(OUTPUT_CSV, index=False, float_format="%.6f")
print(f"Wrote {OUTPUT_CSV} with shape {submission.shape}")
submission.head()


## Notes / ideas to push Hit@3 higher

- **Bigger / better backbone**: swap `CLIP_MODEL_NAME` for `openai/clip-vit-large-patch14` (768-d) or an
  OpenCLIP LAION checkpoint (`open_clip_torch`, e.g. `ViT-L-14` / `laion2b_s32b_b82k`) for stronger
  features — just update `EMBED_DIM` accordingly (the notebook auto-detects the model's real output dim
  in Section 5, so this is a one-line change).
- **Ensembling**: concatenate CLIP embeddings with a self-supervised backbone like DINOv2
  (`facebook/dinov2-base`), which tends to be strong for instance-level retrieval, then L2-normalize the
  concatenation (optionally PCA/whiten down to 512 dims afterward).
- **Stronger TTA for queries**: since queries are cropped/glared/blurred, multi-crop TTA (several crops
  at different scales, not just a horizontal flip) tends to help more than it does for the clean gallery
  images — but we can't tell queries apart from the rest, so this needs to be applied uniformly or
  estimated some other way (e.g. blur/contrast heuristics to guess which images are "query-like").
- **Query expansion / database-side augmentation**: a fully unsupervised technique (e.g. k-reciprocal
  re-ranking) applied to the *embeddings themselves* before submission, since the final CSV format allows
  any post-processing as long as embeddings stay fixed-dimensional and roughly comparable via cosine
  similarity.
